# 00c · Auditoría de políticas (instructor)
> **Workshop Tec de Monterrey · De los datos a los agentes** — Abarrotera del Centro S.A. (empresa ficticia, datos sintéticos).

Comprueba que el gobierno de `03_gobierno` sigue completo después de cualquier cambio (por ejemplo, al agregar una tabla). Solo lee: no modifica nada.

- **Configuración:** ninguna tabla que pueda leer `LECTOR_COMERCIAL` tiene región sin `RAP_REGION`; RAW no es legible; los roles de negocio no tienen permisos directos.
- **Comportamiento:** con `EXECUTE USING POLICY_CONTEXT` se simula cada rol sobre las tablas con RAP y se compara contra lo esperado.

Todo debe salir **PASS**. Si una tabla nueva sale en *con región y sin RAP*, agrégale la política en `03_gobierno` antes de compartirla.

In [ ]:
%%sql -r ctx
-- Contexto de ejecución (primera celda)
USE ROLE TEC_ADMIN;
USE WAREHOUSE TEC_WH;
USE DATABASE TEC_DEMO;

In [ ]:
%%sql -r politicas_config
-- 1) Configuración de permisos y políticas
WITH legibles AS (
  SELECT DISTINCT table_schema, table_name FROM TEC_DEMO.INFORMATION_SCHEMA.TABLE_PRIVILEGES
  WHERE grantee = 'LECTOR_COMERCIAL' AND privilege_type = 'SELECT'),
con_region AS (
  SELECT DISTINCT table_schema, table_name FROM TEC_DEMO.INFORMATION_SCHEMA.COLUMNS WHERE column_name ILIKE '%region%'),
con_rap AS (
  SELECT DISTINCT ref_schema_name table_schema, ref_entity_name table_name
  FROM TABLE(TEC_DEMO.INFORMATION_SCHEMA.POLICY_REFERENCES(POLICY_NAME => 'TEC_DEMO.ANALITICA.RAP_REGION'))),
expuestas AS (
  SELECT l.table_schema || '.' || l.table_name t FROM legibles l JOIN con_region USING (table_schema, table_name)
  LEFT JOIN con_rap r USING (table_schema, table_name) WHERE r.table_name IS NULL),
directos AS (
  SELECT grantee || ':' || table_name g FROM TEC_DEMO.INFORMATION_SCHEMA.TABLE_PRIVILEGES
  WHERE grantee IN ('DIRECTOR_COMERCIAL', 'GERENTE_BAJIO', 'GERENTE_NORTE') AND privilege_type = 'SELECT'),
chequeos AS (
  SELECT 'Tablas legibles con región y sin RAP' chequeo, COALESCE((SELECT LISTAGG(t, ', ') FROM expuestas), '') valor, '' esperado
  -- TEMAS_COMENTARIO no tiene región (solo texto → tema); se lee a través de VOZ_CLIENTE, que sí tiene RAP
  UNION ALL SELECT 'Tablas legibles sin columna de región (solo las documentadas)',
    (SELECT LISTAGG(l.table_name, ', ') WITHIN GROUP (ORDER BY l.table_name) FROM legibles l
     LEFT JOIN con_region c USING (table_schema, table_name) WHERE c.table_name IS NULL), 'TEMAS_COMENTARIO'
  UNION ALL SELECT 'Tablas RAW legibles por LECTOR_COMERCIAL', (SELECT COUNT(*) FROM legibles WHERE table_schema = 'RAW')::STRING, '0'
  UNION ALL SELECT 'SELECT directo a roles de negocio (todo viene de LECTOR_COMERCIAL)', COALESCE((SELECT LISTAGG(g, ', ') FROM directos), ''), ''
  UNION ALL SELECT 'Tablas con RAP_REGION', (SELECT COUNT(*) FROM con_rap)::STRING, '15'
  UNION ALL SELECT 'Columnas con MASK_PII',
    (SELECT COUNT(*) FROM TABLE(TEC_DEMO.INFORMATION_SCHEMA.POLICY_REFERENCES(POLICY_NAME => 'TEC_DEMO.ANALITICA.MASK_PII')))::STRING, '3'
  UNION ALL SELECT 'Roles en ACCESO_REGION',
    (SELECT LISTAGG(rol || '=' || region, ', ') WITHIN GROUP (ORDER BY rol) FROM TEC_DEMO.ANALITICA.ACCESO_REGION),
    'ACCOUNTADMIN=*, DIRECTOR_COMERCIAL=*, GERENTE_BAJIO=Bajío, GERENTE_NORTE=Norte, TEC_ADMIN=*'
)
SELECT IFF(valor = esperado, 'PASS', 'FAIL') estado, chequeo, valor, esperado FROM chequeos ORDER BY estado, chequeo

In [ ]:
import pandas as pd
from snowflake.snowpark.context import get_active_session

# 2) Comportamiento: simula cada rol sobre TODAS las tablas con RAP_REGION y compara las regiones que ve.
# EXECUTE USING POLICY_CONTEXT no se puede usar dentro de un procedimiento, por eso se arma aquí, una consulta por rol.
s = get_active_session()
refs = s.sql("SELECT DISTINCT ref_entity_name, ref_arg_column_names FROM TABLE(TEC_DEMO.INFORMATION_SCHEMA.POLICY_REFERENCES("
             "POLICY_NAME => 'TEC_DEMO.ANALITICA.RAP_REGION')) ORDER BY 1").collect()
union = ' UNION ALL '.join(
    f"SELECT '{r[0]}' t, {__import__('json').loads(r[1])[0]} r FROM TEC_DEMO.ANALITICA.{r[0]}" for r in refs)

TODAS = 'Bajío,Centro,Norte,Sur,Sureste'
ESPERADO = {'GERENTE_BAJIO': 'Bajío', 'GERENTE_NORTE': 'Norte', 'DIRECTOR_COMERCIAL': TODAS,
            'ACCOUNTADMIN': TODAS, 'ROL_INEXISTENTE': ''}  # un rol que no está en ACCESO_REGION no ve nada (fail-closed)

filas = []
for rol, esperado in ESPERADO.items():
    vistas = {r[0]: r[1] for r in s.sql(
        f"EXECUTE USING POLICY_CONTEXT(CURRENT_ROLE => '{rol}') AS "
        f"SELECT t, LISTAGG(DISTINCT r, ',') WITHIN GROUP (ORDER BY r) FROM ({union}) GROUP BY t").collect()}
    for r in refs:
        v = vistas.get(r[0], '')
        filas.append({'estado': 'PASS' if v == esperado else 'FAIL', 'rol': rol, 'tabla': r[0], 've': v, 'esperado': esperado})

res = pd.DataFrame(filas)
print(f"{len(refs)} tablas × {len(ESPERADO)} roles: {(res.estado == 'PASS').sum()} PASS · {(res.estado == 'FAIL').sum()} FAIL")
res[res.estado == 'FAIL'] if (res.estado == 'FAIL').any() else res.groupby('rol').agg(tablas=('tabla', 'count'), ve=('ve', 'first'))

In [ ]:
%%sql -r politicas_mask
-- 3) Enmascaramiento parcial: un gerente ve iniciales del contacto, los últimos 4 dígitos del teléfono y el dominio del correo
--    (ejemplo: "G. C. J." · "******7025" · "****@gmail.com"). El director ve el dato completo.
EXECUTE USING POLICY_CONTEXT(CURRENT_ROLE => 'GERENTE_BAJIO') AS
SELECT IFF(COUNT_IF(NOT (email LIKE '****@%' AND REGEXP_LIKE(telefono, '[*]+[0-9]{4}') AND REGEXP_LIKE(contacto, '([^ ][.] ?)+'))) = 0,
           'PASS', 'FAIL') AS estado,
       COUNT(*) AS filas_revisadas, ANY_VALUE(contacto) AS ejemplo_contacto, ANY_VALUE(telefono) AS ejemplo_telefono, ANY_VALUE(email) AS ejemplo_email
FROM TEC_DEMO.ANALITICA.VENTAS_ENRIQUECIDAS WHERE venta_id < 1000;

In [ ]:
%%sql -r ctx_fin
-- Deja la sesión como estaba
USE ROLE TEC_ADMIN;